In [ ]:
from sklearn import feature_selection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
#data=pd.read_csv('site_num_processed2.csv')
data = pd.read_csv('phedata_merged_mental_cor2.csv')
icd = pd.read_csv(r"prediction_results.csv")
icd.rename(columns={'patient_id':'Participant ID'}, inplace=True)
data = pd.merge(data, icd[['Participant ID', 'risk_score']], on='Participant ID', how='left')
data.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) if i in range(2, 180) else col 
                for i, col in enumerate(data.columns)]
cols = data.columns.tolist()
# if 'any' in cols:
#     cols = ['any'] + [c for c in cols if c != 'any']
#     data = data[cols]
data

In [ ]:
data.columns.tolist()[11:180]

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier
filtered_data = data[(data['Region_Code'] != 1) & (data['Region_Code'] != 9)]
X = filtered_data.filter(data.columns.tolist()[11:180])
y = filtered_data['any']
filtered_data1 = data[(data['Region_Code'] == 1) | (data['Region_Code'] == 9)]
X_test = filtered_data1.filter(data.columns.tolist()[11:180])
y_test = filtered_data1['any']
# sample_weights = compute_sample_weight(
#     class_weight='balanced', 
#     y=y
# )
clf = LGBMClassifier()
clf.fit(X,y)
clf.score(X_test,y_test)
from sklearn import metrics
y_pred = clf.predict_proba(X_test)[:, 1]
auc = round(metrics.roc_auc_score(y_test, y_pred), 3)
auc

In [ ]:
from scipy.stats import reciprocal
from scipy.stats import randint
from scipy.stats.distributions import expon,uniform,norm,poisson,bernoulli,expon,lognorm
import numpy as np
from sklearn.model_selection import  GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import ParameterSampler
from lightgbm import LGBMClassifier

In [ ]:
from lightgbm import LGBMClassifier
clf=LGBMClassifier()

In [ ]:
data.columns.tolist()[11:219]

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import optuna
from optuna.samplers import TPESampler

# 假设data_enc是您的DataFrame，包含特征和标签列
# 假设target_col是您的目标变量列名
# 假设region_col是存储评估中心区域的列名

# 定义评估中心列表（根据您的描述）
regions =  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
region_col = 'Region_Code'
target_col = 'any'

# 确保数据中包含区域信息
assert region_col in data.columns, f"数据中缺少区域列: {region_col}"

# 定义特征列（排除目标列和区域列）
#feature_cols = [col for col in data.columns if col not in ['Participant ID', 'depressed-after','Region_Code']]
feature_cols=data.columns.tolist()[11:180]
# 准备数据
X = data[feature_cols]
y = data[target_col]
groups = data[region_col]  # 用于分组的区域信息

# 创建分组K折交叉验证（10折）
group_kfold = GroupKFold(n_splits=10)

# 定义Optuna目标函数进行超参数优化
def objective(trial):
    # 定义要调优的超参数
    params = {
        'objective': 'binary',  # 假设是二分类问题
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),
        'n_estimators': trial.suggest_int('n_estimators',100, 500),
        'max_depth': trial.suggest_int('max_depth',-1,15),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.7, 1.0),
        'subsample': trial.suggest_float('subsample',0.7, 1.0), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_jobs': 8,
    }
    
    # 存储每折的分数
    fold_scores = []
    
    # 执行留一区交叉验证
    for train_idx, test_idx in group_kfold.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # 创建LightGBM数据集
        train_data = lgb.Dataset(X_train, label=y_train)
        test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
        
        # 训练模型，使用早停防止过拟合
        # 移除了verbose_eval参数，使用callbacks控制日志
        model = lgb.train(
            params,
            train_data,
            valid_sets=[test_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0)  # 禁用训练日志
            ],
            num_boost_round=1000
        )
        
        # 预测并计算分数
        y_pred = model.predict(X_test)
        # 假设是二分类问题，使用AUC作为评估指标
        score = roc_auc_score(y_test, y_pred)
        fold_scores.append(score)
    
    # 返回平均分数
    return np.mean(fold_scores)

# 创建Optuna研究并优化超参数
study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=100)

# 输出最佳参数
print("最佳参数:")
print(study.best_params)
print(f"最佳交叉验证分数: {study.best_value}")

# 使用最佳参数在所有数据上训练最终模型
best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': 1,  # 启用日志
    'boosting_type': 'gbdt',
})

# 执行完整的留一区交叉验证以获得性能估计
final_scores = []
models = []

for train_idx, test_idx in group_kfold.split(X, y, groups=groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
    
    model = lgb.train(
        best_params,
        train_data,
        valid_sets=[test_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50)],
        num_boost_round=1000
    )
    
    y_pred = model.predict(X_test)
    score = roc_auc_score(y_test, y_pred)
    final_scores.append(score)
    models.append(model)
    
    print(f"区域 {groups.iloc[test_idx[0]]} 的测试分数: {score}")

print(f"\n平均交叉验证分数: {np.mean(final_scores):.4f} (±{np.std(final_scores):.4f})")


In [ ]:
lgb.Booster(model_file='model/lightgbm_final_model.txt')

In [ ]:
data

In [ ]:
data.columns.tolist()[10:219]

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import matplotlib as mpl
# 使用 Arial，并嵌入 TrueType 字体到 PDF
mpl.rcParams['font.family'] = ['Arial', 'DejaVu Sans']
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['axes.unicode_minus'] = False  # 负号正常显示

# 假设data是您的DataFrame，包含特征和标签列
# 假设target_col是您的目标变量列名
# 假设region_col是存储评估中心区域的列名

# 定义评估中心列表（根据您的描述）
region_col = 'Region_Code'
target_col = 'any'

# 确保数据中包含区域信息
assert region_col in data.columns, f"数据中缺少区域列: {region_col}"

# 定义特征列（排除目标列和区域列）
feature_cols = data.columns.tolist()[11:219]

# 准备数据
X = data[feature_cols]

y = data[target_col]
groups = data[region_col]  # 用于分组的区域信息

# 创建分组K折交叉验证（10折）
group_kfold = GroupKFold(n_splits=10)

# 存储每折的结果
fold_results = []
models = []
all_fpr = []
all_tpr = []
all_auc = []

# 创建一个DataFrame来存储所有预测结果
all_predictions = pd.DataFrame(index=data.index)
all_predictions['true_label'] = y

# 您已经找到的最佳参数
best_params = {'num_leaves': 27,
 'n_estimators': 479,
 'max_depth': 11,
 'colsample_bytree': 0.7285572090461967,
 'subsample': 0.9477921560138729,
 'learning_rate': 0.05113021337983586,
 'feature_fraction': 0.5455960283724849,
 'bagging_fraction': 0.718091421715654,
 'bagging_freq': 6,
 'min_child_samples': 88,
 'reg_alpha': 3.8759501443283306e-07,
 'reg_lambda': 4.533886009279306,
 'objective': 'binary',
 'metric': 'binary_logloss',
 'verbosity': 1,
 'boosting_type': 'gbdt'}

# 进行10折交叉验证
for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X, y, groups)):
    print(f"\n正在训练第 {fold} 折...")
    
    # 划分训练集和验证集
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # 创建LightGBM数据集
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # 使用最佳参数训练模型
    evals_result = {}
    model = lgb.train(
        best_params,
        train_data,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100),
            lgb.record_evaluation(evals_result)  # 记录评估结果
        ],
        num_boost_round=1000
    )
    
    # 在验证集上评估模型
    y_pred = model.predict(X_val)
    auc = roc_auc_score(y_val, y_pred)
    
    print(f"第 {fold+1} 折结果 - AUC: {auc:.4f}")
    
    # 使用joblib保存模型
    model_filename = f'model/lgbm_fold_{fold}.pkl'
    model_filename = f'model/lgbm_fold_{fold}'
    model.save_model(f'{model_filename}.txt')
    joblib.dump(model, f'{model_filename}.pkl')
    print(f"模型已保存为 {model_filename}")
    
    # 保存验证集的预测结果
    all_predictions.loc[val_idx, f'fold_{fold}_pred'] = y_pred
    
    # 计算ROC曲线
    fpr, tpr, _ = roc_curve(y_val, y_pred)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_auc.append(auc)
    
    # 保存结果
    fold_results.append({
        'fold': fold,
        'auc': auc,
        'region': groups.iloc[val_idx].iloc[0]  # 记录验证集所在的区域
    })
    
    models.append(model)

# 保存所有预测结果
all_predictions['Participant ID'] = data['Participant ID']
all_predictions['Region_Code'] = data['Region_Code']
all_predictions.to_csv('model/all_fold_predictions.csv', index=True)
print("所有折的预测结果已保存到 all_fold_predictions.csv")

# 绘制十折AUC曲线
plt.figure(figsize=(4, 4))

# 绘制每一折的ROC曲线
for i in range(len(all_fpr)):
    plt.plot(all_fpr[i], all_tpr[i], alpha=0.3, linewidth=1,
             label=f'ROC fold {i} (AUC = {all_auc[i]:.3f})')

# 计算平均ROC曲线
mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(len(all_fpr)):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= len(all_fpr)
mean_auc = np.mean(all_auc)

# 计算标准差
std_tpr = np.std([np.interp(mean_fpr, all_fpr[i], all_tpr[i]) for i in range(len(all_fpr))], axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)

# 绘制平均ROC曲线
plt.plot(mean_fpr, mean_tpr, color='b', linestyle='-',
         label=f'Mean ROC (AUC = {mean_auc:.3f} ± {np.std(all_auc):.3f})', linewidth=2)

# 绘制标准差区域
plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=0.2,
                 label='± 1 std. dev.')

# 绘制随机猜测线
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Chance level (AUC = 0.5)')

plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.title('Mean ROC curve with variability', fontsize=11)
plt.legend(loc='lower right', fontsize=7)
# 移除右边和上边边界
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
#plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mean_roc_curve_with_variability.svg', dpi=600, format='svg', facecolor='white', bbox_inches='tight')
plt.show()

# 打印总体结果
print("\n十折交叉验证总体结果:")
for result in fold_results:
    print(f"折 {result['fold']} (区域 {result['region']}): AUC = {result['auc']:.4f}")

final_mean_auc = np.mean([result['auc'] for result in fold_results])
final_std_auc = np.std([result['auc'] for result in fold_results])
print(f"\n平均 AUC: {final_mean_auc:.4f} ± {final_std_auc:.4f}")

# 保存所有结果到文件
results_df = pd.DataFrame(fold_results)
results_df.to_csv('model/cross_validation_results.csv', index=False)
print("详细结果已保存到 cross_validation_results.csv")